In [81]:
# as easy as loading using prior localization code

from one.api import ONE
from pathlib import Path
import yaml
import os
import wfield
import numpy as np
import pandas as pd
from prior_localization.prepare_data import prepare_widefield, prepare_pupil
from brainbox.io.one import SessionLoader
from brainwidemap.bwm_loading import load_trials_and_mask
from prior_localization.functions.utils import compute_mask
from ibl_info.prepare_data_pid import get_new_cinc_intervals
import seaborn as sns
from matplotlib import pyplot as plt
import pickle as pkl
from tqdm import tqdm
from glob import glob
import re
import concurrent.futures
import pickle as pkl
import time
from one.api import ONE
import pandas as pd
from tqdm import tqdm
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from brainbox.singlecell import bin_spikes2D
from iblatlas.regions import BrainRegions
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from ibl_info.utils import check_config, compute_animal_stats
from scipy.ndimage import convolve1d
import traceback
from scipy.stats import zscore

config = check_config()

In [82]:
pupil_csv = pd.read_csv("../data/generated/pupil_stats.csv")

In [83]:
pupil_csv = pupil_csv[~np.isnan(pupil_csv["pupil_z"])]

In [84]:
pupil_csv["signed_contrast"] = pupil_csv["contrastLeft"].fillna(0) - pupil_csv[
    "contrastRight"
].fillna(0)

In [85]:
pupil_csv["difficulty"] = pupil_csv["signed_contrast"].abs()

In [29]:
mask = pupil_csv["signed_contrast"] == 0

In [31]:
pupil_contrast = pupil_csv[~mask]

In [33]:
pupil_contrast["feedback_binary"] = pupil_contrast["feedbackType"].apply(
    lambda x: 1 if x == 1 else 0
)

/var/folders/nr/dw4dxrwd2r3ctns0lnz88jjc0000gn/T/ipykernel_13273/1545550491.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pupil_contrast["feedback_binary"] = pupil_contrast["feedbackType"].apply(


In [35]:
import statsmodels.formula.api as smf

In [87]:
pupil_csv["interaction"] = pupil_csv["pupil_z"] * pupil_csv["difficulty"]

In [43]:
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

In [88]:
df = pupil_csv.copy()

In [95]:
df["feedback_binary"] = df["feedbackType"].apply(lambda x: 1 if x == 1 else 0)

In [96]:
df.groupby("feedbackType").mean("pupil_z")

,probabilityLeft,contrastRight,contrastLeft,pupil_z,signed_contrast,difficulty,interaction,feedback_binary
feedbackType,,,,,,,,
-1.0,0.501784,0.096607,0.096744,0.009354,0.004511,0.096678,0.006589,0.0
1.0,0.508337,0.371470,0.380311,-0.001614,0.011605,0.375975,0.005406,1.0


In [97]:
import warnings

warnings.filterwarnings("ignore")

In [99]:
import statsmodels.formula.api as smf

# Run a Mixed Effect Model (GLMM)
# This gives you p-values, which tell you if the effect is REAL,
# regardless of whether it's strong enough to change the total accuracy.

# We use "pupil_z" and "difficulty" together.
# The coefficient for pupil_z represents the effect of pupil *holding difficulty constant*.
model = smf.logit("feedback_binary ~ pupil_z + difficulty", data=df)
result = model.fit()

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.362787
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:        feedback_binary   No. Observations:                75399
Model:                          Logit   Df Residuals:                    75396
Method:                           MLE   Df Model:                            2
Date:                Tue, 17 Feb 2026   Pseudo R-squ.:                  0.1317
Time:                        16:47:14   Log-Likelihood:                -27354.
converged:                       True   LL-Null:                       -31502.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.7827      0.016     49.303      0.000       0.752       0.814
pupil_z       -0.0296      0.

In [103]:
df["probabilityLeft"]

0        0.5
1        0.5
2        0.5
3        0.5
4        0.5
        ... 
77688    0.2
77689    0.2
77690    0.2
77691    0.2
77692    0.2
Name: probabilityLeft, Length: 75399, dtype: float64